<a href="https://colab.research.google.com/github/CarlosSMWolff/ParamEst-NN/blob/main/notebooks/2-Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training of Neural Networks

##  Goal

In this notebook, we train neural networks (NNs) on simulated photon-counting data for the task of parameter estimation.

This notebook requires a populated `[datapath]\training-trajectories` folder, and will populate a `[datapath]\models` folder.


Each row in the training data $x$ consists on lists of time delays between photo-detection events of the form $D=[\tau_1,\ldots,\tau_N]$.

The target data $y$ consist of the true values of the parameters to be estimated (i.e., the parameters used to simulate the photon-counting data).

This corresponds to $\Delta$ in the 1D parameter estimation case, or a vector with the values $[\Delta,\Omega]$ in the 2D parameter estimation case.

## Imports

In [6]:
import sys
sys.path.insert(0, "..")
import tensorflow as tf
keras = tf.keras
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from random import *
from pathlib import Path
DATAPATH = Path("D:\Dev\ParamEst-NN\ParamEst-NN-main\data")
from tqdm import tqdm

my_seed_number = 42
# Set the random seed for TensorFlow
tf.random.set_seed(my_seed_number)

# Set the random seed for NumPy (used by TensorFlow for certain operations)
np.random.seed(my_seed_number)

### Populate data folder with required data

This notebook requires the following populated folder

- `[datapath]/training-trajectories`.

The folders can be populated with your own data as follows.

- `[datapath]/training-trajectories/`:     Populated by running the notebook [1-Trajectories_generation.ipynb](https://colab.research.google.com/github/CarlosSMWolff/ParamEst-NN/blob/main/1-Trajectories_generation.ipynb)

## Training 1D Models <a id='1D'> </a>

### Load training data (1D $\Delta$ estimation) <a id='load'></a>

In [2]:
path_tau= DATAPATH / "training-trajectories/1D-delta/taus-Delta-1D.npy"
path_param= DATAPATH / "training-trajectories/1D-delta/delta_rand_list-Delta-1D.npy"

In [3]:
tau_list = np.load(path_tau)
Delta_list = np.load(path_param)

ntraj = len(tau_list)

if len(tau_list)!=len(Delta_list):
  print("ERROR: Dimensions of X (tau_list) and Y (Delta_list) do not match!")
else:
  print(f"{ntraj} trajectories loaded")

40000 trajectories loaded


Select the number of trajectories that we want to use for the training (equal or smaller to number of trajectories loaded)

In [4]:
ntraj_select = 40000

tau_list = tau_list[:ntraj_select].astype(np.float32)
Delta_list = Delta_list[:ntraj_select].astype(np.float32)

- We split the data set: 80% training, 20% validation.

- We do not shuffle the data since the trajectories were already generated randomly

In [5]:
njumps = tau_list.shape[1]

# Set data generated from Monte Carlo
X_train_full, y_train_full = tau_list, Delta_list

lenTrain=int(0.8*len(X_train_full))
X_train, X_valid = X_train_full[:lenTrain], X_train_full[lenTrain:]
y_train, y_valid = y_train_full[:lenTrain], y_train_full[lenTrain:]

### Model #1: RNN architecture
<a id='RNN'></a>

In [ ]:
# 选 Δ 网格
delta_min = 0.0
delta_max = 5.0
K = 100
delta_grid = np.linspace(delta_min, delta_max, K)
# 构造整数标签
idx_train = np.rint((y_train - delta_min) / (delta_max - delta_min) * (K - 1)).astype(np.int32)
idx_train = np.clip(idx_train, 0, K-1)

idx_valid = np.rint((y_valid - delta_min) / (delta_max - delta_min) * (K - 1)).astype(np.int32)
idx_valid = np.clip(idx_valid, 0, K-1)

In [43]:
# 定义RNN模型
def create_model():
    dropout = 0.
    activation = "tanh"

    model = keras.models.Sequential([
        keras.layers.LSTM(
            17,
            input_shape=(None, 1),
            return_sequences=True,
            activation=activation,
            dropout=dropout
        ),
        keras.layers.LSTM(
            17,
            return_sequences=False,
            activation=activation,
            dropout=dropout
        ),
        keras.layers.Dense(K, activation="softmax")
    ])
    return model


Training details

In [51]:
#定义新的Loss函数，高斯负对数似然
def gaussian_nll(y_true, y_pred):
    mu = y_pred[:, 0]
    log_var = y_pred[:, 1]
    # 防止爆炸
    log_var = tf.clip_by_value(log_var, -10.0, 10.0)

    return tf.reduce_mean(
        0.5 * (log_var + tf.square(y_true - mu) * tf.exp(-log_var))
    )

epochs = 600
batch_size = 2048

In [ ]:
tf.keras.backend.clear_session()

modelRNN = create_model()

modelRNN.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")]
)

historyRNN = modelRNN.fit(
    np.expand_dims(X_train, axis=-1),
    idx_train,                    
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(
        np.expand_dims(X_valid, axis=-1),
        idx_valid                
    )
)

Epoch 1/600
32/32 [==============================] - 4s 62ms/step - loss: 4.5851 - top5: 0.0640 - val_loss: 4.5168 - val_top5: 0.0820
Epoch 2/600
32/32 [==============================] - 2s 48ms/step - loss: 4.3908 - top5: 0.1022 - val_loss: 4.2652 - val_top5: 0.0950
Epoch 3/600
32/32 [==============================] - 2s 48ms/step - loss: 4.1712 - top5: 0.1094 - val_loss: 4.0821 - val_top5: 0.1258
Epoch 4/600
32/32 [==============================] - 2s 48ms/step - loss: 4.0109 - top5: 0.1517 - val_loss: 3.9470 - val_top5: 0.1716
Epoch 5/600
32/32 [==============================] - 2s 58ms/step - loss: 3.8850 - top5: 0.1881 - val_loss: 3.8273 - val_top5: 0.1974
Epoch 6/600
32/32 [==============================] - 2s 57ms/step - loss: 3.7795 - top5: 0.2078 - val_loss: 3.7311 - val_top5: 0.2074
Epoch 7/600
32/32 [==============================] - 2s 57ms/step - loss: 3.6910 - top5: 0.2226 - val_loss: 3.6542 - val_top5: 0.2250
Epoch 8/600
32/32 [==============================] - 2s 59ms/s

In [ ]:
delta_grid = delta_grid.astype(np.float64)
loss = historyRNN.history["loss"]
val_loss = historyRNN.history["val_loss"]
top5 = historyRNN.history["top5"]
val_top5 = historyRNN.history["val_top5"]
p_val = modelRNN.predict(np.expand_dims(X_valid, axis=-1))
delta_mean = p_val @ delta_grid
delta_map = delta_grid[np.argmax(p_val, axis=1)]
second_moment = p_val @ (delta_grid**2)
delta_std = np.sqrt(np.maximum(second_moment - delta_mean**2, 0.0))

def reg_metrics(y_true, y_pred, name=""):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    err = y_pred - y_true
    mae  = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    bias = float(np.mean(err))
    print(f"{name}MAE  : {mae:.6g}")
    print(f"{name}RMSE : {rmse:.6g}")
    print(f"{name}Bias : {bias:.6g}")
    return mae, rmse, bias

reg_metrics(y_valid, delta_mean, name="posterior-mean ")
reg_metrics(y_valid, delta_map,  name="MAP           ")

# print(p_val.shape)  # (N_val, K)
# print(np.sum(p_val[0]))  # 应该非常接近 1
# print("final val_loss:", historyRNN.history["val_loss"][-1])
# print("final val_top5:", historyRNN.history["val_top5"][-1])
# print("effective classes (train):", np.exp(historyRNN.history["loss"][-1]))
# print("effective classes (val):  ", np.exp(historyRNN.history["val_loss"][-1]))
# print("loss min/max:", np.min(loss), np.max(loss), "has_nan:", np.isnan(loss).any())
# print("val_loss min/max:", np.min(val_loss), np.max(val_loss), "has_nan:", np.isnan(val_loss).any())


250/250 [==============================] - 1s 4ms/step
shapes: (8000, 100) (8000,) (8000,)
posterior-mean MAE  : 0.191419
posterior-mean RMSE : 0.244582
posterior-mean Bias : -0.00717514
MAP           MAE  : 0.208964
MAP           RMSE : 0.276943
MAP           Bias : -0.0868923


(0.20896383436306457, 0.27694330742044815, -0.08689226135447803)

Save the model

In [71]:
modelTrainName = f'model-RNN.h5'
save_dir = DATAPATH / "models/1D/"
save_path = save_dir / modelTrainName
modelRNN.save(save_path)

d:\Dev\Conda\envs\ParamEstNN\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


### Model #2: Hist-Dense architecture

Define the model

In [74]:
# 选 Δ 网格
delta_min = 0.0
delta_max = 5.0
K = 100
delta_grid = np.linspace(delta_min, delta_max, K)
# 构造整数标签
idx_train = np.rint((y_train - delta_min) / (delta_max - delta_min) * (K - 1)).astype(np.int32)
idx_train = np.clip(idx_train, 0, K-1)

idx_valid = np.rint((y_valid - delta_min) / (delta_max - delta_min) * (K - 1)).astype(np.int32)
idx_valid = np.clip(idx_valid, 0, K-1)

In [77]:
from paramest_nn.custom_layers import MyHistogramLayer_Sigmoid

nbins = 128
taumax = 100
width = taumax/nbins

def make_fixed_random_phi(nbins: int, m: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    phi = rng.choice([-1.0, 1.0], size=(nbins, m)).astype(np.float32)
    phi /= np.sqrt(nbins)  # scale stabilize
    return phi

def create_model_Hist(
    nbins=128, taumax=100,
    m=32,                       # 压缩感知测量数
    hidden=(100, 50, 30),
    droprate=0.0,
    seed_phi=0
):
    activation = "relu"

    # Fixed random projection Phi
    Phi = make_fixed_random_phi(nbins, m, seed=seed_phi)

    inputs = keras.Input(shape=(None,), name="click_times_or_taus")

    # 1) Histogram layer -> (batch, nbins)
    h = MyHistogramLayer_Sigmoid(nbins, taumax, trainable=False)(inputs)

    # 做一个简单的归一化，避免“总计数/长度”被直方图幅值混进去
    h_sum = tf.reduce_sum(h, axis=-1, keepdims=True) + 1e-8
    h_norm = h / h_sum

    # 2) Compressive sensing: y = h_norm @ Phi  -> (batch, m)
    y = keras.layers.Dense(
        m,
        use_bias=False,
        trainable=False,
        name="cs_measurement",
        kernel_initializer=keras.initializers.Constant(Phi)
    )(h_norm)

    logN = tf.math.log(h_sum)   # (batch, 1)
    x = tf.concat([y, logN], axis=-1)   # (batch, m+1)

    # 3) Small MLP -> logits over grid
    for i, phi in enumerate(hidden):
        x = keras.layers.Dense(phi, activation=activation, name=f"fc{i+1}")(x)
        if droprate and droprate > 0:
            x = keras.layers.Dropout(droprate, name=f"drop{i+1}")(x)

    logits = keras.layers.Dense(K, name="logits")(x)
    model = keras.Model(inputs=inputs, outputs=logits, name="Hist_CS_Posterior")
    return model

modelHist = create_model_Hist(nbins=nbins, taumax=taumax, m=32)

# ====== compile ======
modelHist.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
    ],
)
# fit
historyHist = modelHist.fit(
    np.expand_dims(X_train, axis=-1),
    idx_train,                    
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(
        np.expand_dims(X_valid, axis=-1),
        idx_valid                
    )
)

Epoch 1/600
16/16 [==============================] - 2s 78ms/step - loss: 4.6067 - acc: 0.0090 - top5: 0.0491 - val_loss: 4.6039 - val_acc: 0.0095 - val_top5: 0.0512
Epoch 2/600
16/16 [==============================] - 1s 66ms/step - loss: 4.6037 - acc: 0.0139 - top5: 0.0540 - val_loss: 4.6027 - val_acc: 0.0198 - val_top5: 0.0632
Epoch 3/600
16/16 [==============================] - 1s 63ms/step - loss: 4.6015 - acc: 0.0195 - top5: 0.0689 - val_loss: 4.5999 - val_acc: 0.0181 - val_top5: 0.0737
Epoch 4/600
16/16 [==============================] - 1s 63ms/step - loss: 4.5959 - acc: 0.0208 - top5: 0.0965 - val_loss: 4.5907 - val_acc: 0.0179 - val_top5: 0.1064
Epoch 5/600
16/16 [==============================] - 1s 63ms/step - loss: 4.5777 - acc: 0.0218 - top5: 0.1072 - val_loss: 4.5581 - val_acc: 0.0176 - val_top5: 0.1029
Epoch 6/600
16/16 [==============================] - 1s 63ms/step - loss: 4.5204 - acc: 0.0217 - top5: 0.1090 - val_loss: 4.4640 - val_acc: 0.0170 - val_top5: 0.1031
Epoc

In [ ]:
delta_grid = delta_grid.astype(np.float64)
loss = historyHist.history["loss"]
val_loss = historyHist.history["val_loss"]
top5 = historyHist.history["top5"]
val_top5 = historyHist.history["val_top5"]

logits = modelHist.predict(np.expand_dims(X_valid, axis=-1))
p = tf.nn.softmax(logits, axis=-1).numpy()
delta_mean = np.sum(p * delta_grid[None, :], axis=1)
idx_map = np.argmax(logits, axis=1)
delta_map = delta_grid[idx_map]
def metrics(pred, name):
        err = pred - y_valid
        mae = np.mean(np.abs(err))
        rmse = np.sqrt(np.mean(err**2))
        bias = np.mean(err)
        print(f"{name:>14} MAE  : {mae:.6f}")
        print(f"{name:>14} RMSE : {rmse:.6f}")
        print(f"{name:>14} Bias : {bias:.6f}")
        return mae, rmse, bias

out_mean = metrics(delta_mean, "post-mean")
out_map  = metrics(delta_map,  "MAP")

p = tf.nn.softmax(logits, axis=-1).numpy()
entropy = -np.sum(p * np.log(p + 1e-12), axis=1)
print("entropy mean:", entropy.mean(), "std:", entropy.std())
delta = delta_grid[None, :]
mean = np.sum(p * delta, axis=1)
second = np.sum(p * (delta**2), axis=1)
var = second - mean**2
std = np.sqrt(np.maximum(var, 0))

cover68 = np.mean((y_valid >= mean-std) & (y_valid <= mean+std))
print("68% cover:", cover68)


250/250 [==============================] - 0s 1ms/step
     post-mean MAE  : 0.247175
     post-mean RMSE : 0.328194
     post-mean Bias : -0.006681
           MAP MAE  : 0.267905
           MAP RMSE : 0.369417
           MAP Bias : -0.010888
entropy mean: 3.1187425 std: 0.34107336
68% cover: 0.69275


Save the model

In [91]:
modelTrainName = f'model-Hist.h5'
save_dir = DATAPATH / "models/1D/"
save_path = save_dir / modelTrainName
modelHist.save(save_path)

d:\Dev\Conda\envs\ParamEstNN\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


### Training Hist-Dense models with noise in the y_train

Define $\sigma$ interval

In [105]:
sigma_min = 0;sigma_max = 0.5;nsigma = 5; sigma_list = np.linspace(sigma_min,sigma_max,nsigma)

Create callback to save the model of the best epoch during training

In [ ]:
file_dir = DATAPATH/"models/1D/noise-y-train/"
def Create_Callback(sigma):
    
    out_dir = Path(file_dir) / "model_best_Hist" / f"sigma_{sigma:.6g}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # .keras
    filepath = out_dir / "best.keras"

    checkpoint = keras.callbacks.ModelCheckpoint(
        filepath=str(filepath),      # 转成 str
        monitor="val_loss",
        verbose=0,
        save_best_only=True,
        mode="min",
    )
    return [checkpoint]


Loop over $\sigma$

In [ ]:
# Δ 网格
delta_min = 0.0
delta_max = 5.0
K = 100
delta_grid = np.linspace(delta_min, delta_max, K).astype(np.float32)

def y_to_idx(y, delta_min, delta_max, K):
    y = np.asarray(y, dtype=np.float32)
    idx = np.rint((y - delta_min) / (delta_max - delta_min) * (K - 1)).astype(np.int32)
    return np.clip(idx, 0, K-1)

def add_noise_and_make_idx(y, sigma, delta_min, delta_max, K, seed=None):
    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=np.float32)
    if sigma > 0:
        y_noisy = y + rng.normal(loc=0.0, scale=sigma, size=y.shape).astype(np.float32)
    else:
        y_noisy = y.copy()
    y_noisy = np.clip(y_noisy, delta_min, delta_max)
    return y_to_idx(y_noisy, delta_min, delta_max, K)

X_train_in = np.expand_dims(X_train, axis=-1).astype(np.float32)
X_valid_in = np.expand_dims(X_valid, axis=-1).astype(np.float32)

tf.keras.backend.clear_session()

modelHist = create_model_Hist()  # 必须输出 Dense(K) logits

# Sparse
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [
    tf.keras.metrics.SparseCategoricalAccuracy(name="acc"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
]

# 每个模型用新的 optimizer（避免 clone + slot state 问题）
def make_optimizer():
    return tf.keras.optimizers.Adam()

modelHist.compile(optimizer=make_optimizer(), loss=loss, metrics=metrics)

# warm start
model_Prev = tf.keras.models.clone_model(modelHist)
model_Prev.compile(optimizer=make_optimizer(), loss=loss, metrics=metrics)

sigma0 = 0.0
idx_train0 = add_noise_and_make_idx(y_train, sigma0, delta_min, delta_max, K, seed=0)
idx_valid0 = add_noise_and_make_idx(y_valid, sigma0, delta_min, delta_max, K, seed=1)

model_Prev.fit(
    X_train_in,
    idx_train0,
    epochs=2,
    batch_size=1280,
    validation_data=(X_valid_in, idx_valid0),
    callbacks=Create_Callback(sigma0),
    verbose=False,
)

# ---- loop over sigma ----
history_list = []

for j, sigma in enumerate(tqdm(sigma_list)):

    model = tf.keras.models.clone_model(model_Prev)
    model.compile(optimizer=make_optimizer(), loss=loss, metrics=metrics)

    idx_train_noise = add_noise_and_make_idx(y_train, sigma, delta_min, delta_max, K, seed=1000 + j)
    idx_valid_noise = add_noise_and_make_idx(y_valid, sigma, delta_min, delta_max, K, seed=2000 + j)

    history = model.fit(
        X_train_in,
        idx_train_noise,
        epochs=epochs,
        batch_size=1280,
        validation_data=(X_valid_in, idx_valid_noise),
        validation_freq=1,
        callbacks=Create_Callback(sigma),
        verbose=False,
    )
    history_list.append(history)


In [ ]:
histories = [pd.DataFrame(result.history) for result in history_list]
val_loss_list = np.array([history.val_loss for history in histories])

plt.plot(sigma_list,val_loss_list[:,-1],'-o', label = "Loss (last epoch)")
plt.plot(sigma_list,np.min(val_loss_list,axis=1),'-ro', label = "Loss (best epoch)")
plt.legend()
plt.xlabel("$\sigma$"); plt.ylabel("")
plt.show()

### Training Hist models with noise in the x_train

Define Callback to save the model of the best epoch during training

In [ ]:
file_dir = DATAPATH /"models/1D/noise-y-train/"

def Create_Callback(sigma):
    
    out_dir = Path(file_dir) / "model_best_Hist" / f"sigma_{sigma:.6g}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # .keras
    filepath = out_dir / "best.keras"

    checkpoint = keras.callbacks.ModelCheckpoint(
        filepath=str(filepath),      # 转成 str
        monitor="val_loss",
        verbose=0,
        save_best_only=True,
        mode="min",
    )
    return [checkpoint]

Loop over $\sigma$

In [ ]:
# Δ 网格
delta_min = 0.0
delta_max = 5.0
K = 100
delta_grid = np.linspace(delta_min, delta_max, K).astype(np.float32)

def x_to_idx(x, delta_min, delta_max, K):
    x = np.asarray(x, dtype=np.float32)
    idx = np.rint((x - delta_min) / (delta_max - delta_min) * (K - 1)).astype(np.int32)
    return np.clip(idx, 0, K-1)

def add_noise_to_X(x, sigma, delta_min, delta_max, K, seed=None):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=np.float32)
    if sigma > 0:
        x_noisy = x + rng.normal(loc=0.0, scale=sigma, size=x.shape).astype(np.float32)
    else:
        x_noisy = x.copy()
    x_noisy = np.clip(x_noisy, delta_min, delta_max)
    return x_to_idx(x_noisy, delta_min, delta_max, K)

X_train_base = X_train.astype(np.float32)
X_valid_base = X_valid.astype(np.float32)

X_train_base = np.expand_dims(X_train_base, axis=-1)
X_valid_base = np.expand_dims(X_valid_base, axis=-1)

tf.keras.backend.clear_session()

modelHist = create_model_Hist()   # Dense(K) logits
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [
    tf.keras.metrics.SparseCategoricalAccuracy(name="acc"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
]

def make_optimizer():
    return tf.keras.optimizers.Adam()

modelHist.compile(optimizer=make_optimizer(), loss=loss, metrics=metrics)

# warm start
model_Prev = tf.keras.models.clone_model(modelHist)
model_Prev.compile(optimizer=make_optimizer(), loss=loss, metrics=metrics)

# ---- warm start (sigma=0, X 不加噪) ----
model_Prev.fit(
    X_train_base,
    idx_train,
    epochs=2,
    batch_size=1280,
    validation_data=(X_valid_base, idx_valid),
    callbacks=Create_Callback(0.0),
    verbose=False,
)

# ---- loop over sigma ----
history_list = []

for j, sigma in enumerate(tqdm(sigma_list)):

    model = tf.keras.models.clone_model(model_Prev)
    model.compile(optimizer=make_optimizer(), loss=loss, metrics=metrics)

    # 只在 X 上加噪声
    X_train_noise = add_noise_to_X(X_train, sigma, seed=1000 + j)
    X_valid_noise = add_noise_to_X(X_valid, sigma, seed=2000 + j)

    X_train_noise = np.expand_dims(X_train_noise, axis=-1)
    X_valid_noise = np.expand_dims(X_valid_noise, axis=-1)

    history = model.fit(
        X_train_noise,
        idx_train,              # 标签不变
        epochs=epochs,
        batch_size=1280,
        validation_data=(X_valid_noise, idx_valid),
        validation_freq=1,
        callbacks=Create_Callback(sigma),
        verbose=False,
    )

    history_list.append(history)


In [ ]:
histories = [pd.DataFrame(result.history) for result in history_list]
val_loss_list = np.array([history.val_loss for history in histories])
plt.plot(sigma_list,val_loss_list[:,-1],'-o')
plt.plot(sigma_list,np.min(val_loss_list,axis=1),'-ro')
#plt.yscale('log')
#plt.xscale('log')
plt.show()

## Training 2D Models <a id='1D'> </a>

Load data

In [7]:
path_tau= DATAPATH/"training-trajectories/2D-delta-omega/taus2D.npy"
path_param= DATAPATH/"training-trajectories/2D-delta-omega/param_rand_list-2D.npy"
L_max = 40
L_choices = tf.constant([10, 15, 20, 30, 40], dtype=tf.int32)
tau_list = np.load(path_tau)[:, :L_max]
param_list = np.load(path_param)
ntraj_select = 40000
tau_list = tau_list[:ntraj_select].astype(np.float32)
param_list = param_list[:ntraj_select].astype(np.float32)

njumps = tau_list.shape[1]

# Set data generated from Monte Carlo
x_train_full, y_train_full = tau_list, param_list

lenTrain=int(0.8*len(x_train_full))
x_train, x_valid = x_train_full[:lenTrain], x_train_full[lenTrain:]
y_train, y_valid = y_train_full[:lenTrain], y_train_full[lenTrain:]


### Model: RFF

define the RFF layer

In [31]:
@keras.saving.register_keras_serializable()
class RFFFeatures(keras.layers.Layer):
    def __init__(self, K=32, w_max=20.0, seed=0, eps=1e-6, use_log=True, **kwargs):
        super().__init__(trainable=False, **kwargs)
        self.K = int(K)
        self.w_max = float(w_max)
        self.seed = int(seed)
        self.eps = float(eps)
        self.use_log = bool(use_log)

    def build(self, input_shape):
        rng = np.random.default_rng(self.seed)
        xi = rng.uniform(-self.w_max, self.w_max, size=(self.K,)).astype(np.float32)
        self.xi = self.add_weight(
            name="xi",
            shape=(self.K,),
            initializer=tf.constant_initializer(xi),
            trainable=False,
        )


    def call(self, taus):
        # taus: (B, T) padded with 0
        mask = tf.cast(taus > 0.0, tf.float32)            # (B, T)
        denom = tf.reduce_sum(mask, axis=1, keepdims=True) + 1e-6  # (B,1)

        x = taus
        if self.use_log:
            x = tf.math.log(x + self.eps)

        phase = tf.expand_dims(x, axis=-1) * self.xi      # (B, T, K)
        c = tf.math.cos(phase) * tf.expand_dims(mask, -1)
        s = tf.math.sin(phase) * tf.expand_dims(mask, -1)

        c_mean = tf.reduce_sum(c, axis=1) / denom         # (B, K)
        s_mean = tf.reduce_sum(s, axis=1) / denom

        return tf.concat([c_mean, s_mean], axis=-1)

    def get_config(self):
        return {"K": self.K, "w_max": self.w_max, "seed": self.seed,
                "eps": self.eps, "use_log": self.use_log}


Define the model

In [ ]:
# 选参数网格
delta_min, delta_max, K1 = 0, 3, 50      # Δ
omega_min, omega_max, K2 = 0.25, 5, 50       # ω

p1_grid = np.linspace(delta_min, delta_max, K1).astype(np.float32)
p2_grid = np.linspace(omega_min, omega_max, K2).astype(np.float32)

p1_tf = tf.constant(p1_grid, dtype=tf.float32)  # (K1,)
p2_tf = tf.constant(p2_grid, dtype=tf.float32)  # (K2,)

def to_grid_index_1d(y, ymin, ymax, K):
    idx = np.rint((y - ymin) / (ymax - ymin) * (K - 1)).astype(np.int32)
    return np.clip(idx, 0, K - 1)

def make_joint_labels_from_param(param, delta_min, delta_max, K1, omega_min, omega_max, K2):
    delta = param[:, 0]
    omega = param[:, 1]
    idx_delta = to_grid_index_1d(delta, delta_min, delta_max, K1)
    idx_omega = to_grid_index_1d(omega, omega_min, omega_max, K2)
    joint = (idx_delta * K2 + idx_omega).astype(np.int32)
    return joint, idx_delta, idx_omega

joint_train, _, _ = make_joint_labels_from_param(
    y_train, delta_min, delta_max, K1, omega_min, omega_max, K2
)
joint_valid, _, _ = make_joint_labels_from_param(
    y_valid, delta_min, delta_max, K1, omega_min, omega_max, K2
)

joint_train = joint_train.astype(np.int32)
joint_valid = joint_valid.astype(np.int32)

L_max_tf = tf.constant(L_max, dtype=tf.int32)

def crop_and_joint(tau, joint):
    L = tf.random.shuffle(L_choices)[0]
    tau_crop = tau[:L]
    return tau_crop, joint

batch_size = 2048

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, joint_train))
    .shuffle(20000)
    .map(crop_and_joint, num_parallel_calls=tf.data.AUTOTUNE)
    .padded_batch(
        batch_size,
        padded_shapes=([None], []),
        padding_values=(tf.constant(0.0, tf.float32), tf.constant(0, tf.int32))
    )
    .prefetch(tf.data.AUTOTUNE)
)

valid_ds = (
    tf.data.Dataset.from_tensor_slices((x_valid, joint_valid))
    .padded_batch(
        batch_size,
        padded_shapes=([None], []),
        padding_values=(tf.constant(0.0, tf.float32), tf.constant(0, tf.int32))
    )
    .prefetch(tf.data.AUTOTUNE)
)


In [10]:
def create_model_RFF2D(
    K1=50, K2=50,
    Kfeat=128, w_max=20.0, seed=0,
    use_log=True,
    add_logN=True,
    activation="relu",
    droprate=0.0,
):
    inp = keras.Input(shape=(None,), name="taus")  # (B, L)

    # RFF features: (B, 2*Kfeat)
    rff = RFFFeatures(K=Kfeat, w_max=w_max, seed=seed, use_log=use_log, name="rff")(inp)

    feats = rff

    # Optional: logN feature for mixed lengths
    if add_logN:
        logN = keras.layers.Lambda(
            lambda t: tf.math.log(tf.reduce_sum(tf.cast(t > 0.0, tf.float32), axis=1, keepdims=True) + 1e-6),
            name="logN"
        )(inp)
        feats = keras.layers.Concatenate(name="concat_feat")([feats, logN])

    # MLP head
    x = keras.layers.Dense(100, activation=activation)(feats)
    x = keras.layers.Dropout(droprate)(x)
    x = keras.layers.Dense(50, activation=activation)(x)
    x = keras.layers.Dropout(droprate)(x)
    x = keras.layers.Dense(30, activation=activation)(x)
    x = keras.layers.Dropout(droprate)(x)
    x = keras.layers.Dense(20, activation=activation)(x)
    x = keras.layers.Dropout(droprate)(x)
    x = keras.layers.Dense(10, activation=activation)(x)
    x = keras.layers.Dropout(droprate)(x)

    logits = keras.layers.Dense(K1 * K2, name="joint_logits")(x)
    return keras.Model(inp, logits, name="RFF_2Dposterior")


modelRFF2D = create_model_RFF2D(
    K1=K1, K2=K2,
    Kfeat=128,        
    w_max=15,      
    seed=0,
    use_log=True,    # 很重要
    add_logN=True,
    droprate=0.0
)


train

In [12]:
tf.keras.backend.clear_session()
modelRFF2D.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)
modelRFF2D.fit(train_ds, validation_data=valid_ds, epochs=100)

Epoch 1/100
16/16 [==============================] - 2s 68ms/step - loss: 7.8237 - val_loss: 7.8228
Epoch 2/100
16/16 [==============================] - 1s 60ms/step - loss: 7.8166 - val_loss: 7.8089
Epoch 3/100
16/16 [==============================] - 1s 65ms/step - loss: 7.7661 - val_loss: 7.7253
Epoch 4/100
16/16 [==============================] - 1s 63ms/step - loss: 7.6250 - val_loss: 7.5694
Epoch 5/100
16/16 [==============================] - 1s 60ms/step - loss: 7.4442 - val_loss: 7.3774
Epoch 6/100
16/16 [==============================] - 1s 61ms/step - loss: 7.2468 - val_loss: 7.1594
Epoch 7/100
16/16 [==============================] - 1s 59ms/step - loss: 7.0635 - val_loss: 6.9329
Epoch 8/100
16/16 [==============================] - 1s 61ms/step - loss: 6.8828 - val_loss: 6.6867
Epoch 9/100
16/16 [==============================] - 1s 61ms/step - loss: 6.7238 - val_loss: 6.4914
Epoch 10/100
16/16 [==============================] - 1s 60ms/step - loss: 6.5810 - val_loss: 6.3284

results

In [13]:
def eval_for_L(model, x_valid, y_valid, p1_grid, p2_grid, L, cred_mass=0.68):
    # 预测
    logits = model.predict(x_valid[:, :L], verbose=0)
    probs = tf.nn.softmax(logits, axis=1).numpy()

    N = probs.shape[0]
    K1, K2 = len(p1_grid), len(p2_grid)
    probs_2d = probs.reshape(N, K1, K2)

    # ---------- entropy / pmax ----------
    eps = 1e-12
    entropy = -np.sum(probs * np.log(probs + eps), axis=1)
    Hmax = np.log(K1 * K2)
    pmax = np.max(probs, axis=1)

    # ---------- posterior mean ----------
    delta_hat = np.sum(probs_2d * p1_grid[:, None], axis=(1, 2))
    omega_hat = np.sum(probs_2d * p2_grid[None, :], axis=(1, 2))

    delta_true = y_valid[:, 0]
    omega_true = y_valid[:, 1]

    delta_rmse = np.sqrt(np.mean((delta_hat - delta_true) ** 2))
    omega_rmse = np.sqrt(np.mean((omega_hat - omega_true) ** 2))
    delta_mae = np.mean(np.abs(delta_hat - delta_true))
    omega_mae = np.mean(np.abs(omega_hat - omega_true))

    # ---------- credible region coverage ----------
    # 先算每个样本的 true grid index
    i_delta = np.argmin(np.abs(delta_true[:, None] - p1_grid[None, :]), axis=1)  # (N,)
    i_omega = np.argmin(np.abs(omega_true[:, None] - p2_grid[None, :]), axis=1)  # (N,)
    true_idx = (i_delta * K2 + i_omega).astype(np.int32)                         # (N,)

    # 对每个样本取 top-k，k 是达到 cred_mass 的最小集合
    coverage = np.zeros(N, dtype=bool)
    for n in range(N):
        p = probs[n]
        order = np.argsort(p)[::-1]
        cum = np.cumsum(p[order])
        cutoff = np.searchsorted(cum, cred_mass)
        kept = order[:cutoff + 1]
        coverage[n] = np.any(kept == true_idx[n])

    coverage = np.mean(coverage)

    # 输出结果
    print("=" * 60)
    print(f"L = {L}   (N={N})")
    print("pmax mean/std/min/max:",
          np.mean(pmax), np.std(pmax), np.min(pmax), np.max(pmax))
    print("entropy mean / Hmax:", np.mean(entropy), Hmax)
    print("entropy ratio:", np.mean(entropy) / Hmax)

    print("\nPosterior mean error:")
    print(f"Δ  RMSE = {delta_rmse:.4e}, MAE = {delta_mae:.4e}")
    print(f"ω  RMSE = {omega_rmse:.4e}, MAE = {omega_mae:.4e}")

    print("\nCalibration:")
    print(f"{int(cred_mass*100)}% credible region coverage:", coverage)

    return {
        "L": L,
        "pmax_mean": float(np.mean(pmax)),
        "entropy_mean": float(np.mean(entropy)),
        "entropy_ratio": float(np.mean(entropy) / Hmax),
        "delta_rmse": float(delta_rmse),
        "omega_rmse": float(omega_rmse),
        "delta_mae": float(delta_mae),
        "omega_mae": float(omega_mae),
        "coverage": float(coverage),
    }

# 跑不同长度
results = []
for L in [10, 15, 20, 30, 40]:
    results.append(eval_for_L(modelRFF2D, x_valid, y_valid, p1_grid, p2_grid, L))

print("\nDone. Collected results for:", [r["L"] for r in results])


L = 10   (N=8000)
pmax mean/std/min/max: 0.013344233 0.012446951 0.0010806686 0.13299619
entropy mean / Hmax: 6.165302 7.824046010856292
entropy ratio: 0.7879940623073377

Posterior mean error:
Δ  RMSE = 7.8575e-01, MAE = 6.5783e-01
ω  RMSE = 7.9773e-01, MAE = 5.2655e-01

Calibration:
68% credible region coverage: 0.547375
L = 15   (N=8000)
pmax mean/std/min/max: 0.01592317 0.012956746 0.0017052676 0.15731998
entropy mean / Hmax: 5.9292088 7.824046010856292
entropy ratio: 0.7578187484155976

Posterior mean error:
Δ  RMSE = 7.5315e-01, MAE = 6.2557e-01
ω  RMSE = 6.3834e-01, MAE = 3.9684e-01

Calibration:
68% credible region coverage: 0.59
L = 20   (N=8000)
pmax mean/std/min/max: 0.018067887 0.013244646 0.002053869 0.15901412
entropy mean / Hmax: 5.7580147 7.824046010856292
entropy ratio: 0.7359382435846514

Posterior mean error:
Δ  RMSE = 7.2795e-01, MAE = 6.0020e-01
ω  RMSE = 5.3167e-01, MAE = 3.1741e-01

Calibration:
68% credible region coverage: 0.61025
L = 30   (N=8000)
pmax mean/st

Save model

In [128]:
modelTrainName = f'model-RFF2D.keras'
save_dir = DATAPATH/"models/2D/"
save_path = save_dir / modelTrainName
modelRFF2D.save(save_path)

#### Fisher information and Cramér–Rao bound